# ML Zoomcamp 2026 — Homework 5: Deployment


In [1]:
import hashlib
import json
import os
import subprocess
import time
from urllib.request import Request, urlopen

# use this folder's uv project, not the repo-level .venv
os.environ.pop('VIRTUAL_ENV', None)

!uv sync --locked -q
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


Check the artifact checksum before unpickling it.

In [11]:
with open('model_metadata.json') as f_in:
    expected = json.load(f_in)['model_sha256']

with open('pipeline.bin', 'rb') as f_in:
    actual = hashlib.sha256(f_in.read()).hexdigest()

assert actual == expected, f'pipeline.bin checksum mismatch: {actual}'
expected

'1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'

## Q1. uv version

In [2]:
!uv --version

uv 0.12.5 (x86_64-unknown-linux-gnu)


## Q2. Locked Scikit-Learn version

In [3]:
!grep scikit-learn pyproject.toml
!grep -A1 '^name = "scikit-learn"' uv.lock

    "scikit-learn==1.7.2",
name = "scikit-learn"
version = "1.7.2"


## Q3. Load the model with pickle

In [4]:
%%writefile q3_load_model.py
import pickle

from model import normalize_record

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

probability = pipeline.predict_proba([normalize_record(lead)])[0, 1]
print(round(float(probability), 3))

Overwriting q3_load_model.py


In [5]:
!uv run python q3_load_model.py

0.533


## Q4. Serve the model with FastAPI
Start `predict:app` with uvicorn in the background, then POST the second lead.

In [6]:
URL = 'http://localhost:9696'
client = {
    'lead_source': 'organic_search',
    'industry': 'technology',
    'employment_status': 'employed',
    'location': 'europe',
    'number_of_courses_viewed': 4,
    'annual_income': 80304.0,
    'interaction_count': 7,
    'lead_score': 0.74,
}

def wait_for_health(url, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urlopen(f'{url}/health') as response:
                return json.load(response)
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f'{url} did not become healthy')

def predict(url, lead):
    request = Request(f'{url}/predict', data=json.dumps(lead).encode(),
                      headers={'Content-Type': 'application/json'}, method='POST')
    with urlopen(request) as response:
        return json.load(response)

In [7]:
server = subprocess.Popen(['uv', 'run', 'uvicorn', 'predict:app', '--host', '0.0.0.0', '--port', '9696'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    print(wait_for_health(URL))
    result = predict(URL, client)
finally:
    server.terminate()
    server.wait()

result, round(result['conversion_probability'], 3)

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


({'conversion_probability': 0.769799, 'conversion': True}, 0.77)

## Q5. Python base image
The tag declared in the checked-in `Dockerfile`:

In [8]:
!head -1 Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


## Q6. Run the container

Build the image and run it in the background, then send the same request as Q4.
It serves the same frozen artifact, so the answer must match Q4.

In [12]:
!docker build -q -t zoomcamp-model:2026-hw5 .
!docker run -d --rm --name hw5 -p 9696:9696 zoomcamp-model:2026-hw5

sha256:ec6844b0913f66aeb08b8beab49f37e971fd532bb7d430cb3477bdf61da0710e
e2b07c0bd59d4fee03a489578b369183b618d2efaca3e58612b9e8c880ea5f47


In [13]:
try:
    health = wait_for_health(URL)
    assert health['status'] == 'ok' and health['model_sha256'] == expected
    print(health)
    result = predict(URL, client)
finally:
    subprocess.run(['docker', 'stop', 'hw5'], capture_output=True)

result, round(result['conversion_probability'], 3)

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


({'conversion_probability': 0.769799, 'conversion': True}, 0.77)